In [0]:
SURVEY_YEAR = 2025
CATALOG = "workspace"
RAW_DIR = f"/Volumes/{CATALOG}/ai_reality_check/raw/survey_{SURVEY_YEAR}"

for layer in ["rc_bronze", "rc_silver", "rc_gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{layer}")

print("Source :", RAW_DIR)
print([r.databaseName for r in spark.sql(f"SHOW SCHEMAS IN {CATALOG}").collect()])

Source : /Volumes/workspace/ai_reality_check/raw/survey_2025
['ai_reality_check', 'default', 'information_schema', 'rc_bronze', 'rc_gold', 'rc_silver']


In [0]:
from pyspark.sql import functions as F

df = (spark.read
      .option("header", True)
      .option("multiLine", True)
      .option("quote", '"')
      .option("escape", '"')
      .option("inferSchema", False)
      .csv(f"{RAW_DIR}/results.csv"))

stats = df.agg(
    F.count("*").alias("lignes"),
    F.sum(F.col("ResponseId").isNull().cast("int")).alias("responseid_null"),
    F.countDistinct("ResponseId").alias("responseid_distincts"),
).collect()[0]

print("Colonnes :", len(df.columns), "(attendu : 172)")
print("Lignes :", stats["lignes"])
print("ResponseId null :", stats["responseid_null"])
print("ResponseId distincts :", stats["responseid_distincts"])
print("Colonnes parasites (_c...) :", [c for c in df.columns if c.startswith("_c")])

Colonnes : 172 (attendu : 172)
Lignes : 49191
ResponseId null : 0
ResponseId distincts : 49191
Colonnes parasites (_c...) : []


In [0]:
df.select("ResponseId", "MainBranch", "Age", "Employment").show(5, truncate=False)
print([c for c in df.columns if c.startswith("AI")])


+----------+------------------------------+---------------+----------------------------------------------------+
|ResponseId|MainBranch                    |Age            |Employment                                          |
+----------+------------------------------+---------------+----------------------------------------------------+
|1         |I am a developer by profession|25-34 years old|Employed                                            |
|2         |I am a developer by profession|25-34 years old|Employed                                            |
|3         |I am a developer by profession|35-44 years old|Independent contractor, freelancer, or self-employed|
|4         |I am a developer by profession|35-44 years old|Employed                                            |
|5         |I am a developer by profession|35-44 years old|Independent contractor, freelancer, or self-employed|
+----------+------------------------------+---------------+-------------------------------------

In [0]:
import re, uuid, hashlib
from datetime import datetime, timezone
from pyspark.sql import functions as F

SURVEY_YEAR = 2025
CATALOG = "workspace"
RAW_DIR = f"/Volumes/{CATALOG}/ai_reality_check/raw/survey_{SURVEY_YEAR}"
SRC_FILE = f"{RAW_DIR}/results.csv"
BRONZE_TABLE = f"{CATALOG}.rc_bronze.survey_results"
COLMAP_TABLE = f"{CATALOG}.rc_bronze.column_map"
LOG_TABLE = f"{CATALOG}.rc_bronze.ingestion_log"

EXPECTED = {2025: {"size": 140893245,
                   "sha256": "2d1f65308877282edfb4470520eabbc08cb499118432a3dcec6a66c086aa2baa"}}
RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc).replace(tzinfo=None)

# 1. Intégrité du fichier source (échec critique si différent)
h, size = hashlib.sha256(), 0
with open(SRC_FILE, "rb") as f:
    while chunk := f.read(1024 * 1024):
        h.update(chunk); size += len(chunk)
exp = EXPECTED[SURVEY_YEAR]
assert size == exp["size"] and h.hexdigest() == exp["sha256"], "CRITIQUE : fichier source différent de la référence"
print(f"Fichier source OK : {size:,} octets, SHA-256 conforme")

# 2. Lecture brute : tout en texte
df = (spark.read.option("header", True).option("multiLine", True)
      .option("quote", '"').option("escape", '"').option("inferSchema", False)
      .csv(SRC_FILE))

# 3. Normalisation des noms de colonnes
def sanitize(name):
    return re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_")

orig_cols = df.columns
new_cols = [sanitize(c) for c in orig_cols]
dups = sorted({c for c in new_cols if new_cols.count(c) > 1})
assert not dups, f"CRITIQUE : noms en collision après normalisation : {dups}"
changed = [(o, n) for o, n in zip(orig_cols, new_cols) if o != n]
print(f"{len(changed)} noms de colonnes normalisés")

# 4. Contrôles critiques
df_b = (df.toDF(*new_cols)
          .withColumn("_survey_year", F.lit(SURVEY_YEAR))
          .withColumn("_source_file", F.lit(SRC_FILE))
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_batch_id", F.lit(RUN_ID)))
s = df_b.agg(F.count("*").alias("n"),
             F.sum(F.col("ResponseId").isNull().cast("int")).alias("nulls"),
             F.countDistinct("ResponseId").alias("distincts")).collect()[0]
rows_read = s["n"]
assert len(orig_cols) == 172, f"CRITIQUE : {len(orig_cols)} colonnes au lieu de 172"
assert s["nulls"] == 0 and s["distincts"] == rows_read, "CRITIQUE : ResponseId nul ou dupliqué"
assert not [c for c in orig_cols if c.startswith("_c")], "CRITIQUE : colonnes parasites"
print(f"Contrôles critiques OK : {rows_read:,} lignes, {len(orig_cols)} colonnes")

# 5. Colonnes d'IA, une par ligne (pour planifier Silver)
for c in [c for c in orig_cols if c.startswith("AI")]:
    print(" -", c)

Fichier source OK : 140,893,245 octets, SHA-256 conforme
16 noms de colonnes normalisés
Contrôles critiques OK : 49,191 lignes, 172 colonnes
 - AILearnHow
 - AIThreat
 - AIModelsChoice
 - AIModelsHaveWorkedWith
 - AIModelsWantToWorkWith
 - AIModelsAdmired
 - AIModelsHaveEntry
 - AIModelsWantEntry
 - AISelect
 - AISent
 - AIAcc
 - AIComplex
 - AIToolCurrently partially AI
 - AIToolDon't plan to use AI for this task
 - AIToolPlan to partially use AI
 - AIToolPlan to mostly use AI
 - AIToolCurrently mostly AI
 - AIFrustration
 - AIExplain
 - AIAgents
 - AIAgentChange
 - AIAgent_Uses
 - AIAgentImpactSomewhat agree
 - AIAgentImpactNeutral
 - AIAgentImpactSomewhat disagree
 - AIAgentImpactStrongly agree
 - AIAgentImpactStrongly disagree
 - AIAgentChallengesNeutral
 - AIAgentChallengesSomewhat disagree
 - AIAgentChallengesStrongly agree
 - AIAgentChallengesSomewhat agree
 - AIAgentChallengesStrongly disagree
 - AIAgentKnowledge
 - AIAgentKnowWrite
 - AIAgentOrchestration
 - AIAgentOrchWrite
 

In [0]:
log_schema = ("run_id string, survey_year int, source_file string, file_size long, "
              "sha256 string, rows_read long, rows_written long, n_columns int, "
              "status string, message string, started_at timestamp, finished_at timestamp")
status, message, rows_written = "FAILED", "", None
try:
    (df_b.write.format("delta").mode("overwrite")
         .option("replaceWhere", f"_survey_year = {SURVEY_YEAR}")
         .saveAsTable(BRONZE_TABLE))

    colmap = spark.createDataFrame(
        [(SURVEY_YEAR, i, o, n) for i, (o, n) in enumerate(zip(orig_cols, new_cols), 1)],
        "survey_year int, ordinal int, original_name string, bronze_name string")
    (colmap.write.format("delta").mode("overwrite")
           .option("replaceWhere", f"survey_year = {SURVEY_YEAR}")
           .saveAsTable(COLMAP_TABLE))

    rows_written = spark.table(BRONZE_TABLE).where(f"_survey_year = {SURVEY_YEAR}").count()
    assert rows_written == rows_read, f"CRITIQUE : {rows_read} lues, {rows_written} écrites"
    status = "SUCCESS"
except Exception as e:
    message = str(e)[:500]
    raise
finally:
    row = [(RUN_ID, SURVEY_YEAR, SRC_FILE, size, h.hexdigest(), rows_read, rows_written,
            len(orig_cols), status, message, STARTED_AT,
            datetime.now(timezone.utc).replace(tzinfo=None))]
    spark.createDataFrame(row, log_schema).write.format("delta").mode("append").saveAsTable(LOG_TABLE)

print(f"{status} : {rows_read:,} lignes lues, {rows_written:,} écrites dans {BRONZE_TABLE}")
display(spark.table(LOG_TABLE).orderBy(F.col("started_at").desc()))

SUCCESS : 49,191 lignes lues, 49,191 écrites dans workspace.rc_bronze.survey_results


run_id,survey_year,source_file,file_size,sha256,rows_read,rows_written,n_columns,status,message,started_at,finished_at
468d92a3-bc0d-4d04-9e28-902a1c8e2ca8,2025,/Volumes/workspace/ai_reality_check/raw/survey_2025/results.csv,140893245,2d1f65308877282edfb4470520eabbc08cb499118432a3dcec6a66c086aa2baa,49191,49191,172,SUCCESS,,2026-10-25T14:04:09.805Z,2026-10-25T14:04:46.587Z


In [0]:
from pyspark.sql import functions as F

CATALOG, Y = "workspace", 2025
b = spark.table(f"{CATALOG}.rc_bronze.survey_results").where(f"_survey_year = {Y}")

print("Lignes Bronze 2025 :", b.count(), "| ResponseId distincts :", b.select("ResponseId").distinct().count())
print("Colonnes Bronze :", len(b.columns), "(172 sources + 4 métadonnées = 176 attendues)")
print("column_map :", spark.table(f"{CATALOG}.rc_bronze.column_map").where(f"survey_year = {Y}").count(), "lignes (172 attendues)")
spark.sql(f"DESCRIBE DETAIL {CATALOG}.rc_bronze.survey_results").select("format", "numFiles", "sizeInBytes").show(truncate=False)
spark.table(f"{CATALOG}.rc_bronze.ingestion_log").select("status", "rows_read", "rows_written", "started_at").orderBy("started_at").show(truncate=False)

# Texte officiel des questions clés (schema.csv) + distribution brute des réponses
sch = (spark.read.option("header", True).option("multiLine", True)
       .option("quote", '"').option("escape", '"')
       .csv(f"/Volumes/{CATALOG}/ai_reality_check/raw/survey_{Y}/schema.csv"))
for c in ["AISelect", "AISent", "AIAcc"]:
    print(f"\n===== {c} =====")
    sch.where(F.col("qname") == c).select("question", "type").show(truncate=False)
    b.groupBy(c).count().orderBy(F.desc("count")).show(truncate=False)

Lignes Bronze 2025 : 49191 | ResponseId distincts : 49191
Colonnes Bronze : 176 (172 sources + 4 métadonnées = 176 attendues)
column_map : 172 lignes (172 attendues)
+------+--------+-----------+
|format|numFiles|sizeInBytes|
+------+--------+-----------+
|delta |1       |7417709    |
+------+--------+-----------+

+-------+---------+------------+--------------------------+
|status |rows_read|rows_written|started_at                |
+-------+---------+------------+--------------------------+
|SUCCESS|49191    |49191       |2026-10-25 14:04:09.805362|
+-------+---------+------------+--------------------------+


===== AISelect =====
+----------------------------------------------------------+----+
|question                                                  |type|
+----------------------------------------------------------+----+
|Do you currently use AI tools in your development process?|MC  |
+----------------------------------------------------------+----+

+----------------------------